# Домашнее задание № 10. Машинный перевод

## Задание 1 (6 баллов + 2 доп балла).
Попробуйте обучить трансформерную переводную модель также как в семинаре, но вместо пары языков возьмите несколько исходных языков сразу и совместите их в одной модели. Например, создайте модель EN, FR, DE -> RU (можно использовать другие языки).
Для того, чтобы обучить такую модель вам нужно будет собрать датасет из нескольких парных корпусов. Можете просто склеить все в одну большую выборку, а можете взять какую-то выборку из единичных корпусов, чтобы общее количество данных не было таким большим.
Используйте как минимум 3 исходных языка.
Не забудьте переобучить токенизаторы на новых датасетах!

Можно использовать как основу любую из реализаций из семинара (MultiHeadAttention, nn.Transformer, global + local attention).
Параметры ниже точно работают в колабе и модель обучается достаточно быстро. Попробуйте их немного увеличить (batch size возможно придется наоборот уменьшить). Обучайте модель хотя бы 5 эпох, а желательно больше, чтобы тестовые примеры начали переводиться более менее адекватно.

После обучения возьмите хотя бы по 50 примеров на язык из тестовой части корпуса и переведите их. Оцените качество переводов с помощью метрики BLEU и ChrF для каждого из языков.
Обязательно посмотрите на получаемые переводы! Если все переводы это пустые строки или зацикленные повторы одного символа, то пообучайте модель подольше.
Найдите лучшие (как минимум по 2 на язык) переводы согласно этими метрикам и проверьте действительно ли они хорошие.
В качестве дополнительного теста проверьте может ли модель перевести пару предложений на каком-то другом хотя бы немного близком языке (например, если в обучающих исходных текстах вы использовали EN, FR, IT то попробуйте переводить испанский). Получаемый результат совсем неправильный или модель смогла найти и использовать какие-то общие закономерности?



Чтобы получить 2 доп балла вам нужно будет придумать как оптимизировать функцию translate. Сейчас она работает только с одним текстом - это не эффективно. Можно генерировать переводы сразу для нескольких текстов (батча). Главная сложность с таким подходом состоит в том, что генерируемые тексты будут заканчиваться в разное время и нужно сделать столько итераций, сколько нужно для завершения всех текстов (т.е. условие на то, что последний токен не равен [EOS] в текущем коде не сработает).
ВАЖНО - недостаточно просто изменить входной аргумент с text на texts и добавить еще один цикл по texts! Сама модель должна вызываться на нескольких текстах! Функция с batch prediction должна работать быстрее, поэтому переведите всю тестовую выборку и оцените качество BLEU как минимум на 2000 примеров (а лучше на всем тестовом корпусе).

In [ ]:
# параметры которые работат в колабе
embed_dim = 32
num_heads = 4
ff_dim = embed_dim*2
num_layers = 2
batch_size = 400


## Задание 2 (2 балла).
Прочитайте главу про машинный перевод у Журафски и Маннига - https://web.stanford.edu/~jurafsky/slp3/13.pdf
Ответьте своими словами в чем заключается техника back translation? Для чего она применяется и что позволяет получить? Опишите по шагам как ее применить к паре en->ru на данных из семинара. Сколько моделей понадобится? Сколько запусков обучения нужно будет сделать?

Ответ должен содержать как минимум 10 предложений.

Суть метода заключается в создании синтетических параллельных данных. Для этого используется уже обученная модель перевода, но работает она в обратном направлении. Если наша цель — улучшить модель для перевода с английского на русский, мы берем большой корпус текстов только на русском языке. Затем с помощью предобученной модели, переводящей с русского на английский, мы генерируем для этих русских текстов их английские соответствия. В результате мы получаем искусственную параллельную пару: (синтетический английский, оригинальный русский), которую можно добавить к основным обучающим данным.

Чтобы применить эту технику к паре en->ru, используя данные из семинара , нужно выполнить следующие шаги:

1) Загрузить параллельные данные en-ru из OPUS-100 для обучения базовой модели.

2) Обучить первую модель машинного перевода ru->en на этих параллельных данных.

3) Найти или использовать дополнительные тексты только на русском языке (монолингвальный корпус), которые не входят в параллельную выборку.

4) С помощью обученной на шаге 2 модели ru->en перевести все дополнительные русские тексты на английский язык. Важно, что полученный английский текст будет неидеальным, шумным, но это и есть нужный нам синтетический источник.

5) Объединить исходные "чистые" параллельные данные en->ru с новыми синтетическими данными (сгенерированный en -> оригинальный ru). В итоге получается расширенный тренировочный корпус.

6) На этом расширенном наборе данных обучить итоговую модель для перевода с английского на русский en->ru.

Понадобятся модели, реализующие два направления перевода. Однако физически это могут быть как две отдельные модели (одна для ru->en, другая для en->ru), так и одна двунаправленная модель (что реже для RNN-архитектур, но возможно). В классическом подходе это будут две отдельные модели. Итоговая целевая модель (en->ru) — это третья, если считать отдельно, но по сути, это финальная модель, которую мы и хотим получить.

Запуски:
Первый запуск: обучение вспомогательной модели ru->en на исходных параллельных данных.

Второй запуск: обучение (или дообучение) целевой модели en->ru на расширенном наборе данных, включающем синтетику.

Таким образом потребуется как минимум два полноценных цикла обучения. Количество моделей зависит от того, как их считать, но процесс включает две отдельные обучающие процедуры, что в конечном итоге позволяет получить одну улучшенную модель перевода с английского на русский.